In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import *
from sklearn.preprocessing import StandardScaler, MinMaxScaler

In [2]:
# python -m pip install --upgrade certifi truststore
import requests
from io import BytesIO
import certifi
path = 'https://bit.ly/advertising_csv'
# data1 = pd.read_csv(path)
# data1.head()
response = requests.get(
    path,
    verify=certifi.where(),
    timeout=15
)

response.raise_for_status()

data1 = pd.read_csv(BytesIO(response.content))

print(data1.head())

      TV  Radio  Newspaper  Sales
0  230.1   37.8       69.2   22.1
1   44.5   39.3       45.1   10.4
2   17.2   45.9       69.3    9.3
3  151.5   41.3       58.5   18.5
4  180.8   10.8       58.4   12.9


In [3]:
# data1 데이터 프레임을 매체 와 매출 로 나눠서 저장

target = 'Sales'
x = data1.drop(target, axis=1) # Sales 열 drop 한 데이터프레임
y = data1.loc[:, target] # Sales 열의 모든 행 데이터 저장
print(x.head())
print(y.head())

      TV  Radio  Newspaper
0  230.1   37.8       69.2
1   44.5   39.3       45.1
2   17.2   45.9       69.3
3  151.5   41.3       58.5
4  180.8   10.8       58.4
0    22.1
1    10.4
2     9.3
3    18.5
4    12.9
Name: Sales, dtype: float64


In [ ]:
# x와 y의 데이터를 학습 데이터와 검증 데이터로 split
x_train, x_val, y_train, y_val = train_test_split(x, y, test_size= .3, random_state= 20)
# test_size= .3 은 train : test = 7 : 3 로 나누겠다는 의미
# random_state = 20 은 무작위 값들에 규칙을 부여하여 일관성 있는 결과값 확보

In [5]:
# scaler로 데이터 스케일링
scaler = MinMaxScaler() # 모든 값을 0과 1사이의 값으로 변환
x_train = scaler.fit_transform(x_train) # 학습 데이터의 최대(1)최소(0)를 기준(fit)하여 변환
x_val = scaler.transform(x_val) # 학습 데이터의 최대최소를 기준으로 검증 데이터 변환

In [6]:
# Modeling

from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(x_train, y_train) # xy_train를 이용해 fit 메서드로 학습
pred = model.predict(x_val)  # x_val을 이용해 predict 메서드로 예측, 검증은 나중에 y_val로
# pred는 R의 predict( result , newdata = test )와 유사

In [7]:
# 목표값(y_val) 과 예측값(pred)의 오차를 활용해 성능 계산
print('RMSE:', root_mean_squared_error(y_val, pred))
print('R2:', r2_score(y_val, pred))
print('MAE:', mean_absolute_error(y_val,pred))
print('MAPE:', mean_absolute_percentage_error(y_val, pred))

RMSE: 2.1824586788058977
R2: 0.8568542870042719
MAE: 1.5030135035019552
MAPE: 0.23452930250376364


In [8]:
# 분류 모델링 예제(로지스틱 선형회귀)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import *
from sklearn.preprocessing import StandardScaler, MinMaxScaler

In [10]:
import requests
from io import BytesIO
import certifi

path = "https://bit.ly/graduate_csv"

response = requests.get(
    path,
    verify = certifi.where(),
    timeout=15
)

response.raise_for_status()

data2 = pd.read_csv(BytesIO(response.content))

data2.head()

,admit,gre,gpa,rank
0,0,380,3.61,3
1,1,660,3.67,3
2,1,800,4.00,1
3,1,640,3.19,4
4,0,520,2.93,4


In [12]:
target = 'admit'
x = data2.drop(target, axis=1)
y = data2.loc[:, target]

In [ ]:
cat_cols = ['rank'] # 범주형 가변수 Rank
x = pd.get_dummies(x, columns= cat_cols, drop_first= True) 
# 범주형 변수를 get_dummies를 통해 가변수(0혹은1)형태로 변환
# drop_first= True 첫번째 범주를 제거해 다중공선성 문제 방지
x.head()

,gre,gpa,rank_2,rank_3,rank_4
0,380,3.61,False,True,False
1,660,3.67,False,True,False
2,800,4.00,False,False,False
3,640,3.19,False,False,True
4,520,2.93,False,False,True


In [15]:
x_train, x_val, y_train, y_val = train_test_split(x, y, test_size= .3, random_state= 20)
scaler = MinMaxScaler()
x_train = scaler.fit_transform(x_train)
x_val = scaler.transform(x_val)

In [16]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression()
model.fit(x_train, y_train)
pred = model.predict(x_val)

In [17]:
# 분류모델 평가( 혼동 행렬 )
print(confusion_matrix(y_val, pred))

[[74 10]
 [26 10]]


In [18]:
# 평가 결과 정리
# 분류 보고서 함수
print(classification_report(y_val, pred))

              precision    recall  f1-score   support

           0       0.74      0.88      0.80        84
           1       0.50      0.28      0.36        36

    accuracy                           0.70       120
   macro avg       0.62      0.58      0.58       120
weighted avg       0.67      0.70      0.67       120

